# Advanced Transformations — Declarative (Zero-Code)

This notebook showcases the **four new advanced transformation blocks** added to `ds_engine.preparation`:

| Block | `type:` in YAML | Purpose |
|---|---|---|
| Power Transform | `power_transform` | Yeo-Johnson or Box-Cox — normalises skew |
| Outlier Clipping | `clip_outliers` | IQR or Z-score winsorisation |
| Log Transform | `log_transform` | `log`, `log1p`, or `sqrt` |
| Binning | `bin` | Equal-width or quantile discretisation |

Following DSEngine's **Zero-Code** philosophy, the entire pipeline is declared in a YAML file.  
No custom Python is needed to run any transformation.

---


## 1 — The YAML Configuration

Everything is declared here. No Python needed.

In [1]:
with open('../configs/07_advanced_transforms.yml', 'r') as f:
    print(f.read())

07_advanced_transforms_demo:
  data:
    source: "examples/datasets/prep_demo.csv"

  output:
    path: "examples/outputs/"
    format: ["json", "html"]

  steps:
    # â”€â”€ Step 1: Fill Missing Values first (prerequisite for downstream steps) â”€â”€
    - name: "fill_missing_values"
      type: "impute"
      columns: ["age", "income"]
      params:
        method: "median"

    # â”€â”€ Step 2: Clip extreme values with IQR winsorisation â”€â”€
    - name: "clip_outliers_iqr"
      type: "clip_outliers"
      columns: ["age", "income", "score"]
      params:
        method: "iqr"
        factor: 1.5

    # â”€â”€ Step 3: Log1p to compress the right-skewed 'income' column â”€â”€
    - name: "log_income"
      type: "log_transform"
      columns: ["income"]
      params:
        method: "log1p"

    # â”€â”€ Step 4: Yeo-Johnson power transform on remaining numeric â”€â”€
    - name: "power_transform_yj"
      type: "power_transform"
      columns: ["age", "score"]
      params:
      

## 2 — Peek at the Raw Dataset

In [2]:
import sys, os
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

df_raw = pd.read_csv('../examples/datasets/prep_demo.csv')
print('Shape:', df_raw.shape)
print('\nMissing values:')
print(df_raw.isnull().sum())
print('\nSkewness (raw):')
print(df_raw.select_dtypes('number').skew().round(3))
df_raw

Shape: (20, 5)

Missing values:
age          1
income       2
group        0
score        0
education    0
dtype: int64

Skewness (raw):
age       0.143
income   -0.134
score     0.000
dtype: float64


,age,income,group,score,education
0,25.0,50000.0,A,10,High School
1,NaN,60000.0,A,12,Bachelor
2,30.0,NaN,B,11,Master
3,35.0,80000.0,B,15,Bachelor
4,40.0,90000.0,C,14,PhD
5,22.0,45000.0,A,9,High School
6,28.0,55000.0,B,13,Bachelor
7,33.0,75000.0,C,16,Master
8,38.0,85000.0,A,11,PhD
9,45.0,95000.0,B,14,Master


## 3 — Run the Pipeline

One call. That's it.

In [3]:
from ds_engine.utils import pipeline_runner

config_path     = '../configs/07_advanced_transforms.yml'
experiment_name = '07_advanced_transforms_demo'

pipeline_runner.run(config_path, experiment_name)

Dataset has fewer than 30 rows. This may affect the reliability of statistical tests.


+--------------------------------------------------------------+
|  DSEngine  |  Experiment: 07_advanced_transforms_demo   |
|  Config  : ../configs/07_advanced_transforms.yml           |
|  Started : 2026-03-25 08:02:10                             |
+--------------------------------------------------------------+
> Loading data...
  Source  : D:\OneDrive - STEPLESMOSENSESARL\PlesmoSense-CENTAN\Code\ACHRAF_Private\DS_Engine\examples\datasets\prep_demo.csv
  Shape   : 20 rows x 5 columns
  Columns : age, income, group, score, education
> Pre-flight inspection...
  v  No fatal issues.
  !  1 warning(s) — check before interpreting results:
     [1] Dataset has fewer than 30 rows. This may affect the reliability of statistical tests.
> Running steps (7 total):
  [1/7] fill_missing_values (impute        ) ... done  (0.00s)
  [2/7] clip_outliers_iqr (clip_outliers ) ... done  (0.01s)
  [3/7] log_income       (log_transform ) ... done  (0.00s)
  [4/7] power_transform_yj (power_transform) ... 

0

## 4 — Load the Report

In [4]:
import glob, json

reports = sorted(
    glob.glob('../examples/outputs/07_advanced_transforms_demo/*/report.json'),
    reverse=True
)
latest = reports[0]
print('Report:', latest)

with open(latest) as f:
    report = json.load(f)

print('\nSteps recorded:', list(report['steps'].keys()))

Report: ../examples/outputs/07_advanced_transforms_demo\20260325_080210\report.json

Steps recorded: ['pre_flight_inspection', 'fill_missing_values', 'clip_outliers_iqr', 'log_income', 'power_transform_yj', 'bin_age', 'bin_income_quantile', 'post_transform_summary']


### 4a — Outlier Clipping (IQR Winsorisation)

In [5]:
clip = report['steps']['clip_outliers_iqr']
print(f"Method: {clip['method']}   Factor: {clip['factor']}\n")

rows = []
for col, info in clip['clipped_columns'].items():
    rows.append({
        'column'         : col,
        'lower_bound'    : round(info['lower_bound'], 2),
        'upper_bound'    : round(info['upper_bound'], 2),
        'clipped_low'    : info['n_clipped_low'],
        'clipped_high'   : info['n_clipped_high'],
        'total_clipped'  : info['total_clipped'],
    })

pd.DataFrame(rows)

Method: iqr   Factor: 1.5



,column,lower_bound,upper_bound,clipped_low,clipped_high,total_clipped
0,age,16.0,50.0,0,0,0
1,income,17875.0,122875.0,0,0,0
2,score,6.5,18.5,0,0,0


### 4b — Log Transform (income → log1p)

In [6]:
log = report['steps']['log_income']
print(f"Method: {log['method']}\n")

rows = []
for col, info in log['transformed_columns'].items():
    rows.append({
        'column'      : col,
        'skew_before' : info['skew_before'],
        'skew_after'  : info['skew_after'],
        'reduction_%' : round((1 - abs(info['skew_after']) / max(abs(info['skew_before']), 1e-9)) * 100, 1),
    })

pd.DataFrame(rows)

Method: log1p



,column,skew_before,skew_after,reduction_%
0,income,-0.2445,-0.4636,-89.6


### 4c — Power Transform (Yeo-Johnson)

The optimal `λ` is fitted per-column using Maximum Likelihood. `λ ≈ 1` means no transform needed; `λ ≈ 0` approximates a log transform.

In [7]:
pt = report['steps']['power_transform_yj']
print(f"Method: {pt['method']}\n")

rows = []
for col, info in pt['transformed_columns'].items():
    rows.append({
        'column'      : col,
        'lambda (λ)'  : info['lambda'],
        'skew_before' : info['skew_before'],
        'skew_after'  : info['skew_after'],
    })

pd.DataFrame(rows)

Method: yeo-johnson



,column,lambda (λ),skew_before,skew_after
0,age,0.5104,0.1526,-0.0245
1,score,0.7885,0.0000,-0.0459


### 4d — Binning (uniform & quantile)

In [8]:
for step_name in ['bin_age', 'bin_income_quantile']:
    bstep = report['steps'][step_name]
    print(f"[{step_name}]  method={bstep['method']}  n_bins={bstep['n_bins']}")
    for col, info in bstep['binned_columns'].items():
        print(f"  Original: {col}  →  New column: {info['new_column']}")
        print(f"  Bin counts:")
        for label, cnt in info['bin_counts'].items():
            print(f"    {label}: {cnt}")
    print()

[bin_age]  method=uniform  n_bins=4
  Original: age  →  New column: agegroup
  Bin counts:
    young: 4
    adult: 5
    senior: 7
    elder: 4

[bin_income_quantile]  method=quantile  n_bins=4
  Original: income  →  New column: income_quartile
  Bin counts:
    Q1_low: 5
    Q2_mid: 6
    Q3_high: 4
    Q4_top: 5



---

## Summary — What each block does

| Block | `type:` | Key params | Effect |
|---|---|---|---|
| Outlier Clipping | `clip_outliers` | `method`: `iqr`\|`zscore`, `factor` | Winsorises extreme values |
| Log Transform | `log_transform` | `method`: `log`\|`log1p`\|`sqrt` | Compresses right skew |
| Power Transform | `power_transform` | `method`: `yeo-johnson`\|`box-cox` | Auto-normalises any skew |
| Binning | `bin` | `method`: `uniform`\|`quantile`, `n_bins`, `labels` | Discretises continuous columns |

> **Chaining is automatic.** Each step receives the transformed DataFrame from the previous step,  
> enabling complex multi-step preparation pipelines with a single YAML declaration.
